In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs/convert/parquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs/convert/parquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs/convert/parquet/default/test/0000.parquet"
}

ds = load_dataset("parquet", data_files=data_files)

print(ds)
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


default/train/0000.parquet:   0%|          | 0.00/475k [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/63.3k [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 11426
    })
    validation: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 1583
    })
    test: Dataset({
        features: ['sentence', 'sentiment', 'topic'],
        num_rows: 3166
    })
})
{'sentence': 'slide giáo trình đầy đủ .', 'sentiment': 2, 'topic': 1}


In [ ]:
!pip install transformers torch pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.0 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer
from pyvi import ViTokenizer
from torch.utils.data import Dataset, DataLoader

In [ ]:
MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        # 1. Lấy dòng dữ liệu thô
        row = self.dataset[index]
        text = row['sentence']  # Cột chứa câu bình luận
        label = row['sentiment'] # Cột chứa nhãn (0, 1, 2)

        # 2. TÁCH TỪ (Bắt buộc cho PhoBERT)
        # "Giảng viên nhiệt tình" -> "Giảng_viên nhiệt_tình"
        text_segmented = ViTokenizer.tokenize(text)

        # 3. Tokenize & Encoding (Chuyển chữ thành số ID)
        encoding = self.tokenizer.encode_plus(
            text_segmented,
            add_special_tokens=True,    # Thêm token đặc biệt <s> và </s>
            max_length=self.max_len,
            padding='max_length',       # Thêm số 1 cho đủ độ dài
            truncation=True,            # Cắt bớt nếu quá dài
            return_attention_mask=True, # Tạo mask để model biết chỗ nào là padding
            return_tensors='pt',        # Trả về PyTorch Tensor
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# --- TẠO DATALOADER ---

# Tạo Dataset Object từ biến 'ds' bạn đã load
train_ds = SentimentDataset(ds['train'], tokenizer, MAX_LEN)
val_ds = SentimentDataset(ds['validation'], tokenizer, MAX_LEN)
test_ds = SentimentDataset(ds['test'], tokenizer, MAX_LEN)

# Tạo DataLoader (Cái này mới là cái đưa vào model train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# --- KIỂM TRA KẾT QUẢ ---
# Lấy thử 1 batch ra xem đã đúng format chưa
batch = next(iter(train_loader))
print("Kích thước Input IDs:", batch['input_ids'].shape)       # Mong đợi: [32, 128]
print("Kích thước Labels:", batch['labels'].shape)             # Mong đợi: [32]
print("Mẫu dữ liệu đã mã hóa:", batch['input_ids'][0])

Kích thước Input IDs: torch.Size([32, 128])
Kích thước Labels: torch.Size([32])
Mẫu dữ liệu đã mã hóa: tensor([   0,  960, 5415,  592,  563,    4, 3857,    5,    2,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1])


Building Model & Training Loop

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW  # <--- SỬA LẠI DÒNG NÀY (Lấy từ torch.optim)

# --- CÁC PHẦN CÒN LẠI GIỮ NGUYÊN ---

# 1. Cấu hình thiết bị (Ưu tiên GPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang sử dụng thiết bị: {device}")

# 2. Load Model PhoBERT
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

# 3. Khai báo Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# 4. Hàm tính độ chính xác
def calculate_accuracy(preds, labels):
    pred_flat = torch.argmax(preds, dim=1).flatten()
    labels_flat = labels.flatten()
    return torch.sum(pred_flat == labels_flat) / len(labels_flat)

# --- VÒNG LẶP HUẤN LUYỆN (Copy lại phần dưới để chạy liền mạch) ---
EPOCHS = 3

for epoch in range(EPOCHS):
    print(f'\n======== Epoch {epoch + 1} / {EPOCHS} ========')

    # Train
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        model.zero_grad()

        outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()

        if step % 50 == 0 and step > 0:
            print(f"  Batch {step}  - Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"  Average Training Loss: {avg_train_loss:.4f}")

    # Validation
    print("  Đang chạy Validation...")
    model.eval()
    total_eval_accuracy = 0

    for batch in val_loader:
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)

        logits = outputs.logits
        total_eval_accuracy += calculate_accuracy(logits, b_labels).item()

    avg_val_accuracy = total_eval_accuracy / len(val_loader)
    print(f"  Validation Accuracy: {avg_val_accuracy:.4f}")

print("\nĐã huấn luyện xong!")

Đang sử dụng thiết bị: cuda


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



======== Epoch 1 / 3 ========


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

  Batch 50  - Loss: 0.2279
  Batch 100  - Loss: 0.2320
  Batch 150  - Loss: 0.3060
  Batch 200  - Loss: 0.1825
  Batch 250  - Loss: 0.0321
  Batch 300  - Loss: 0.1637
  Batch 350  - Loss: 0.1179
  Average Training Loss: 0.2767
  Đang chạy Validation...
  Validation Accuracy: 0.9468

======== Epoch 2 / 3 ========
  Batch 50  - Loss: 0.0900
  Batch 100  - Loss: 0.3317
  Batch 150  - Loss: 0.1502
  Batch 200  - Loss: 0.1367
  Batch 250  - Loss: 0.0447
  Batch 300  - Loss: 0.0259
  Batch 350  - Loss: 0.1324
  Average Training Loss: 0.1602
  Đang chạy Validation...
  Validation Accuracy: 0.9505

======== Epoch 3 / 3 ========
  Batch 50  - Loss: 0.0186
  Batch 100  - Loss: 0.1390
  Batch 150  - Loss: 0.2184
  Batch 200  - Loss: 0.1091
  Batch 250  - Loss: 0.1328
  Batch 300  - Loss: 0.0255
  Batch 350  - Loss: 0.2212
  Average Training Loss: 0.1228
  Đang chạy Validation...
  Validation Accuracy: 0.9467

Đã huấn luyện xong!


In [ ]:
# Tạo thư mục lưu model
output_dir = './phobert_sentiment_model'
import os
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Lưu model và tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model đã được lưu tại {output_dir}")

Model đã được lưu tại ./phobert_sentiment_model


Evaluation in Testing

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Hàm dự đoán trên tập Test
def get_predictions(model, data_loader):
    model = model.eval() # Chế độ đánh giá

    predictions = []
    real_values = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1) # Lấy nhãn có xác suất cao nhất

            predictions.extend(preds.cpu().tolist())
            real_values.extend(labels.cpu().tolist())

    return predictions, real_values

# 2. Chạy dự đoán
print("Đang đánh giá trên tập Test...")
y_preds, y_test = get_predictions(model, test_loader)

# 3. In báo cáo chi tiết
class_names = ['Tiêu cực', 'Trung tính', 'Tích cực'] # Tương ứng nhãn 0, 1, 2
print("\nBáo cáo phân loại (Classification Report):")
print(classification_report(y_test, y_preds, target_names=class_names))

Đang đánh giá trên tập Test...

Báo cáo phân loại (Classification Report):
              precision    recall  f1-score   support

    Tiêu cực       0.94      0.97      0.96      1409
  Trung tính       0.75      0.55      0.63       167
    Tích cực       0.95      0.96      0.95      1590

    accuracy                           0.94      3166
   macro avg       0.88      0.83      0.85      3166
weighted avg       0.94      0.94      0.94      3166



In [ ]:
def predict_sentiment(text):
    # 1. Xử lý dữ liệu giống hệt lúc train
    text_segmented = ViTokenizer.tokenize(text)
    encoded_review = tokenizer.encode_plus(
        text_segmented,
        max_length=128,
        add_special_tokens=True,
        return_token_type_ids=False,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )

    # 2. Đưa vào model
    input_ids = encoded_review['input_ids'].to(device)
    attention_mask = encoded_review['attention_mask'].to(device)

    output = model(input_ids, attention_mask)
    _, prediction = torch.max(output.logits, dim=1)

    # 3. Map số sang chữ
    class_names = ['Tiêu cực 😡', 'Trung tính 😐', 'Tích cực 😃']
    return class_names[prediction.item()]

# --- CHẠY THỬ NGAY TẠI ĐÂY ---
print("Kết quả 1:", predict_sentiment("Giảng viên dạy quá chán, buồn ngủ."))
print("Kết quả 2:", predict_sentiment("Thầy nhiệt tình, slide đẹp, rất thích môn này."))
print("Kết quả 3:", predict_sentiment("Phòng học bình thường, điều hòa hơi lạnh."))

Kết quả 1: Tiêu cực 😡
Kết quả 2: Tích cực 😃
Kết quả 3: Trung tính 😐


In [ ]:
import os

# Tạo thư mục lưu
save_path = './phobert_sentiment_final'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Lưu model và tokenizer
print(f"Đang lưu model vào {save_path}...")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Đã lưu thành công!")

# (Tùy chọn) Nén lại để tải về máy local cho dễ
!zip -r phobert_sentiment_final.zip ./phobert_sentiment_final

Đang lưu model vào ./phobert_sentiment_final...
Đã lưu thành công!
  adding: phobert_sentiment_final/ (stored 0%)
  adding: phobert_sentiment_final/special_tokens_map.json (deflated 57%)
  adding: phobert_sentiment_final/added_tokens.json (stored 0%)
  adding: phobert_sentiment_final/tokenizer_config.json (deflated 77%)
  adding: phobert_sentiment_final/config.json (deflated 52%)
  adding: phobert_sentiment_final/vocab.txt (deflated 55%)
  adding: phobert_sentiment_final/model.safetensors (deflated 17%)
  adding: phobert_sentiment_final/bpe.codes (deflated 59%)
